# Color to Greyscale

Convert an input image to greyscale using GDAL.

In [1]:
# Papermill parameters cell -- values here are overridden at execution time.
input_image = "nasa_maap_logo.png"
output_file = "greyscale.tif"

In [2]:
# Parameters
input_image = "/var/lib/cwl/stg435d4f6e-e038-46f8-ad30-51e374197909/nasa_maap_logo.png"
output_file = "greyscale.tif"


In [3]:
import os

import numpy as np
from osgeo import gdal

gdal.UseExceptions()

In [4]:
def color_to_greyscale(input_image, output_file, output_dir="output"):
    """Convert ``input_image`` to a single-band greyscale image.

    The result is written to ``output_dir/output_file`` so the OGC workflow
    can collect the output directory.
    """
    os.makedirs(output_dir, exist_ok=True)
    output_path = os.path.join(output_dir, output_file)

    src = gdal.Open(input_image)
    if src is None:
        raise FileNotFoundError(f"Could not open input image: {input_image}")

    xsize, ysize = src.RasterXSize, src.RasterYSize

    if src.RasterCount >= 3:
        # Interpret the first three bands as R, G, B.
        r = src.GetRasterBand(1).ReadAsArray().astype("float32")
        g = src.GetRasterBand(2).ReadAsArray().astype("float32")
        b = src.GetRasterBand(3).ReadAsArray().astype("float32")
        grey = 0.299 * r + 0.587 * g + 0.114 * b
    else:
        # Already single-band -- pass it through.
        grey = src.GetRasterBand(1).ReadAsArray().astype("float32")

    grey = grey.round().clip(0, 255).astype("uint8")

    driver = gdal.GetDriverByName("GTiff")
    dst = driver.Create(output_path, xsize, ysize, 1, gdal.GDT_Byte)

    # Preserve geospatial metadata when the input has any.
    dst.SetGeoTransform(src.GetGeoTransform())
    projection = src.GetProjection()
    if projection:
        dst.SetProjection(projection)

    dst.GetRasterBand(1).WriteArray(grey)
    dst.FlushCache()

    # Release datasets so the file is fully written to disk.
    dst = None
    src = None

    return output_path

In [5]:
output_path = color_to_greyscale(input_image, output_file)
print(f"Wrote greyscale image to {output_path}")

Wrote greyscale image to output/greyscale.tif
